# Essay-only AES training on Kaggle GPU

Attach the pinned AES runtime Dataset with Add Input, select GPU T4 x2 and enable Internet, then Run All.
The training cell exposes **EPOCHS = 20**, **BATCH_SIZE = 16** (global) and **USE_TWO_GPUS = True**.
These settings match the accepted full-train bundle reviewed on 24 October 2026.
This is a saved reproduction notebook; the original executed notebook was not included in the results ZIP.

The model trains on all 8,236 prepared train essays and reports on all 1,635 test essays.
Outputs are four integers 0–9 in dataset order. Input is right-truncated to **512 tokens including special tokens**;
truncation is recorded. Task Response is an essay-only proxy. Judge, SFT, PANDA and database jobs are separate.
The encoder is FacebookAI/roberta-base at the immutable revision in code/offline_research/config.yaml.
No validation or early stopping is used; test labels are used only for reporting within this run.

ZIP and unpacked Dataset inputs are detected automatically and checked against the pinned inventory.
Each bootstrap creates a new working directory. Training outputs are saved under a new run name.
The accepted bundle, original ZIP and evaluation report are stored with the local research artifacts.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid
import zipfile

def prepare_project(input_root: str | Path, working_root: str | Path,
                    bundle_path: str | Path | None = None, expected_sha256: str | None = None,
                    *, expected_files: dict[str, str] | None = None) -> Path:
    """Verify an AES runtime bundle and copy it into a new working directory.

    Args:
        input_root (str | Path): Directory to search when no bundle is specified.
        working_root (str | Path): Parent for the new project directory.
        bundle_path (str | Path | None): Explicit ZIP or unpacked project path.
        expected_sha256 (str | None): Optional digest for a ZIP input.
        expected_files (dict[str, str] | None): Relative file paths and digests.
            Required for an unpacked input.

    Returns:
        Path: New project directory containing the verified runtime.

    Raises:
        ValueError: The input is ambiguous, unsafe or fails verification.
        FileNotFoundError: The input or a required runtime file is missing.
    """
    input_root = Path(input_root)
    if bundle_path:
        candidates = [Path(bundle_path)]
    else:
        candidates = []
        for path in input_root.rglob("*.zip"):
            if zipfile.is_zipfile(path):
                with zipfile.ZipFile(path) as archive:
                    if "code/shared/paths.yaml" in archive.namelist():
                        candidates.append(path)
        if not candidates:
            candidates = sorted({path.parent.parent.parent for path in input_root.rglob("paths.yaml")
                                 if path.parent.name == "shared" and path.is_file()})
    if len(candidates) != 1:
        visible = sorted(str(path) for path in input_root.iterdir())[:20] if input_root.is_dir() else []
        raise ValueError(
            f"Expected one AES ZIP or unpacked project. Found: {candidates}. "
            f"Visible inputs: {visible}. Attach the AES runtime Dataset with Add Input, "
            "or set bundle_path to its ZIP or project directory."
        )
    source = candidates[0]
    if not source.exists():
        raise FileNotFoundError(f"Input source does not exist: {source}")
    if source.is_file():
        digest = hashlib.sha256(source.read_bytes()).hexdigest()
        if expected_sha256 and digest != expected_sha256:
            raise ValueError("Runtime ZIP SHA256 does not match this notebook release")
    elif not expected_files:
        raise ValueError("Unpacked input requires the expected_files hashes from the updated notebook")
    project = Path(working_root) / ("ielts-aes-" + uuid.uuid4().hex[:10])
    project.mkdir(parents=True, exist_ok=False)
    if source.is_file():
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                if not (project / member.filename).resolve().is_relative_to(project.resolve()):
                    raise ValueError("Unsafe archive path")
            archive.extractall(project)
    else:
        source = source.resolve()
        if not (source / "code/offline_research/TrainAes.py").is_file():
            raise FileNotFoundError("Input is missing code/offline_research/TrainAes.py; attach the AES runtime Dataset")
        for name in expected_files:
            original = (source / name).resolve()
            target = (project / name).resolve()
            if not original.is_relative_to(source) or not target.is_relative_to(project.resolve()):
                raise ValueError(f"Unsafe input path: {name}")
            if not original.is_file():
                raise FileNotFoundError(f"AES training input file is missing: {original}")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(original, target)
    if expected_files:
        for name, expected in expected_files.items():
            path = (project / name).resolve()
            if not path.is_relative_to(project.resolve()) or not path.is_file():
                raise ValueError(f"AES training runtime file is missing or unsafe: {name}")
            if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
                raise ValueError(f"AES training runtime file SHA256 mismatch: {name}")
    if not (project / "code/offline_research/TrainAes.py").is_file():
        raise FileNotFoundError("AES training bundle must include code/offline_research/TrainAes.py")
    print("Input source:", source)
    return project


In [ ]:
bundle_path = None
expected_sha256 = None
expected_files = None
raise ValueError("Build and import the matching AESTraining.ipynb with Manage.py research bundle-aes")
project_root = prepare_project("/kaggle/input", "/kaggle/working", bundle_path, expected_sha256,
                               expected_files=expected_files)
os.environ["HF_HOME"] = str(project_root / ".cache/huggingface")
os.environ["HF_HUB_CACHE"] = str(project_root / ".cache/huggingface/hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("Project:", project_root)
print((project_root / "code/offline_research/config.yaml").read_text())


The next cell creates a venv without ensurepip, then installs pip into that venv using the notebook's pip.
The venv retains access to Kaggle's CUDA PyTorch. Transformers/Hub are pinned to the project versions;
only missing data dependencies are added. CUDA torch is constrained to the supplied version.
Actual package versions and all GPU names are exported. The public encoder does not require an HF token.


In [ ]:
import importlib.metadata as metadata
import venv

runtime_dir = project_root / (".venv-" + uuid.uuid4().hex[:8])
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(runtime_dir)
runtime_python = runtime_dir / "bin/python"
# Seed pip locally without calling ensurepip in Kaggle's Python 3.13.
subprocess.run([sys.executable, "-m", "pip", "--python", str(runtime_python),
                "install", "--disable-pip-version-check", "--ignore-installed", "--no-deps",
                "pip==" + metadata.version("pip")], check=True)
requirements = {}
for line in (project_root / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "==" in line:
        requirements[line.split("==", 1)[0]] = line
required_data = ("numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "PyYAML")
missing = []
for name in required_data:
    try:
        metadata.version(name)
    except metadata.PackageNotFoundError:
        missing.append(requirements[name])
constraint = project_root / "torch-constraint.txt"
constraint.write_text("torch==" + metadata.version("torch") + chr(10), encoding="utf-8")
subprocess.run([str(runtime_python), "-m", "pip", "install", "--disable-pip-version-check",
                "--constraint", str(constraint), requirements["transformers"],
                requirements["huggingface-hub"], *missing], check=True)
environment_script = '''
import importlib.metadata as metadata, json, platform, torch
from pathlib import Path
assert torch.cuda.is_available(), "Select GPU in Kaggle Settings"
packages = ["torch", "transformers", "huggingface-hub", "tokenizers", "numpy", "pandas",
            "pyarrow", "scipy", "scikit-learn", "PyYAML"]
record = {"python": platform.python_version(), "cuda": torch.version.cuda,
          "gpu": torch.cuda.get_device_name(0), "gpu_count": torch.cuda.device_count(),
          "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
          "versions": {name: metadata.version(name) for name in packages}}
Path("Environment.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))
'''
subprocess.run([str(runtime_python), "-B", "-c", environment_script], cwd=project_root, check=True)


In [ ]:
# Configure the next training run here. BATCH_SIZE is shared across both GPUs.
EPOCHS = 20
BATCH_SIZE = 16
USE_TWO_GPUS = True

run_dir = project_root / "results" / f"aes-e{EPOCHS}-b{BATCH_SIZE}-{uuid.uuid4().hex[:6]}"
training_script = '''
import json, shutil, sys
from pathlib import Path
import torch
import yaml

project, output = map(Path, sys.argv[1:3])
settings = json.loads(sys.argv[3])
for name in ("epochs", "batch_size"):
    if type(settings[name]) is not int or settings[name] < 1:
        raise ValueError(f"{name} must be a positive integer")
if type(settings["use_two_gpus"]) is not bool:
    raise ValueError("use_two_gpus must be boolean")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator in Kaggle Settings")
devices = [0, 1] if settings["use_two_gpus"] else [0]
if torch.cuda.device_count() < len(devices):
    raise RuntimeError("Select GPU T4 x2, or explicitly set USE_TWO_GPUS = False")
if settings["batch_size"] < len(devices):
    raise ValueError("Global batch size must be at least the number of GPUs")
sys.path.insert(0, str(project / "code"))
from ielts_judge.aes.OrdinalRegression import OrdinalRegression
import TrainAes as aes

config = yaml.safe_load((project / "code/offline_research/config.yaml").read_text(encoding="utf-8"))
config["aes"].update(epochs=settings["epochs"], batch_size=settings["batch_size"])
config_path = project / f"RunConfig-{output.name}.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
original_fit = OrdinalRegression.fit

def fit_on_gpus(self, texts, y_ord, epochs=3, batch_size=8):
    self.device = torch.device("cuda:0")
    base = self.model.to(self.device)
    self.model = torch.nn.DataParallel(base, device_ids=devices) if len(devices) > 1 else base
    try:
        original_fit(self, texts, y_ord, epochs=epochs, batch_size=batch_size)
    finally:
        # Save the underlying model, so the bundle also loads on one GPU or CPU.
        self.model = base
    self.training_metadata.update(
        parallelism="DataParallel" if len(devices) > 1 else "single_gpu",
        gpu_count=len(devices),
        gpu_names=[torch.cuda.get_device_name(i) for i in devices],
        batch_size_scope="global")

OrdinalRegression.fit = fit_on_gpus
sys.argv = ["TrainAes.py", "train", "--require-cuda", "--config", str(config_path),
            "--output", str(output)]
aes.main()
shutil.copyfile(config_path, output / "run_config.yaml")
'''
subprocess.run([str(runtime_python), "-u", "-B", "-c", training_script,
                str(project_root), str(run_dir),
                json.dumps({"epochs": EPOCHS, "batch_size": BATCH_SIZE,
                            "use_two_gpus": USE_TWO_GPUS})],
               cwd=project_root, check=True)
print((run_dir / "aes_report.json").read_text(encoding="utf-8"))


In [ ]:
# Edit this independent essay before running the fresh-process smoke.
new_essay = (
    "Public libraries remain valuable even when information is available online. "
    "They provide quiet places to study and help people who cannot afford reliable internet access. "
    "A library can also teach readers how to distinguish evidence from misleading claims. "
    "Digital services should therefore complement libraries rather than replace them. "
    "Local authorities should adapt opening hours and provide access to electronic resources, "
    "while keeping printed books for readers who prefer them. "
    "The strongest argument for libraries is equal access to education."
)
essay_path = project_root / "new_essay.txt"
essay_path.write_text(new_essay, encoding="utf-8")
smoke = subprocess.run([str(runtime_python), "-B", str(project_root / "code/offline_research/TrainAes.py"),
                        "predict", "--bundle", str(run_dir / "aes_bundle"),
                        "--essay-file", str(essay_path), "--device", "cpu"],
                       cwd=project_root, env={**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"},
                       capture_output=True, text=True, check=True)
(run_dir / "new_essay_smoke.json").write_text(smoke.stdout, encoding="utf-8")
print(smoke.stdout)
# Export model, provenance, reports and environment together; no training caches.
with zipfile.ZipFile(project_root / "aes_training_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(run_dir))
    archive.write(project_root / "Environment.json", "Environment.json")
    archive.write(essay_path, "new_essay.txt")
print("Download:", project_root / "aes_training_results.zip")


The output ZIP contains the complete AES bundle, experiment manifest, ID-aligned test predictions,
metrics and train-median baselines, timing/truncation reports, run_config.yaml, Environment.json,
and the independent essay with its fresh-process offline prediction.

The accepted 20-epoch run has already passed the local AES acceptance review. Re-running this notebook creates
a new experiment; exact learned weights and metrics may differ with CUDA and package versions.
The original Kaggle run did not record peak RAM/VRAM. Local contract tests do not replace a new hosted GPU run.
